# Hand Gesture Recognition (ML vs Native MediaPipe)

This notebook explores two ways to recognize gestures: using a custom Scikit-Learn model and using MediaPipe's built-in Gesture Recognizer.

## 1. Import Dependencies

In [ ]:
import cv2
import mediapipe as mp
import time
import numpy as np
import joblib
import pandas as pd

## 2. Setup CUSTOM ML Model (Section 3)

In [ ]:
# Configurations for the ML model
try:
    custom_model = joblib.load('gesture_classifier.joblib')
    COLUMNS = []
    for i in range(21):
        COLUMNS.extend([f'x{i}', f'y{i}', f'z{i}'])
    print("Custom model loaded successfully!")
except:
    print("WARNING: 'gesture_classifier.joblib' not found. Run 'train_model.py' first.")

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8),
    (9, 10), (10, 11), (11, 12), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20), (5, 9), (9, 13), (13, 17)
]

## 3. Webcam Feed with CUSTOM ML Model

Uses landmarks extracted by MediaPipe and predicts using our Scikit-Learn model.

In [ ]:
options_ml = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1
)

with HandLandmarker.create_from_options(options_ml) as landmarker:
    cap = cv2.VideoCapture(0)
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
            
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        result = landmarker.detect(mp_image)
        
        h, w, _ = frame.shape
        if result.hand_landmarks:
            for hand_landmarks in result.hand_landmarks:
                # Prepare input for ML
                X_raw = []
                for lm in hand_landmarks:
                    X_raw.extend([lm.x, lm.y, lm.z])
                
                X_raw = np.array(X_raw)
                wrist_x, wrist_y, wrist_z = X_raw[0], X_raw[1], X_raw[2]
                X_norm = X_raw.copy()
                for j in range(0, 63, 3):
                    X_norm[j] -= wrist_x
                    X_norm[j+1] -= wrist_y
                    X_norm[j+2] -= wrist_z
                
                df_input = pd.DataFrame([X_norm], columns=COLUMNS)
                prediction = custom_model.predict(df_input)[0]
                proba = custom_model.predict_proba(df_input)
                
                # Drawing
                for connection in HAND_CONNECTIONS:
                    start_pt = (int(hand_landmarks[connection[0]].x * w), int(hand_landmarks[connection[0]].y * h))
                    end_pt = (int(hand_landmarks[connection[1]].x * w), int(hand_landmarks[connection[1]].y * h))
                    cv2.line(frame, start_pt, end_pt, (0, 255, 0), 2)
                
                display_text = f"ML: {prediction.upper()} ({np.max(proba):.2f})"
                cv2.putText(frame, display_text, (int(hand_landmarks[0].x * w), int(hand_landmarks[0].y * h) - 20), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
        
        cv2.imshow('ML Custom Recognition', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'): break
            
    cap.release()
    cv2.destroyAllWindows()

## 4. Webcam Feed with NATIVE MediaPipe Model

Uses built-in Gesture Recognizer from MediaPipe (gesture_recognizer.task).

In [ ]:
options_native = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path='gesture_recognizer.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1
)

with GestureRecognizer.create_from_options(options_native) as recognizer:
    cap = cv2.VideoCapture(0)
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
            
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        result = recognizer.recognize(mp_image)
        
        h, w, _ = frame.shape
        if result.hand_landmarks:
            for i, hand_landmarks in enumerate(result.hand_landmarks):
                # skeleton drawing
                for connection in HAND_CONNECTIONS:
                    start_pt = (int(hand_landmarks[connection[0]].x * w), int(hand_landmarks[connection[0]].y * h))
                    end_pt = (int(hand_landmarks[connection[1]].x * w), int(hand_landmarks[connection[1]].y * h))
                    cv2.line(frame, start_pt, end_pt, (255, 0, 0), 2)
                
                # Get native gesture
                if result.gestures and len(result.gestures) > i:
                    top_gesture = result.gestures[i][0]
                    text = f"NATIVE: {top_gesture.category_name} ({top_gesture.score:.2f})"
                    cv2.putText(frame, text, (int(hand_landmarks[0].x * w), int(hand_landmarks[0].y * h) - 20), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        
        cv2.imshow('Native MediaPipe Recognition', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'): break
            
    cap.release()
    cv2.destroyAllWindows()